In [1]:
import sys

sys.path.append('..')

from src.graph_loader import load_graph, load_all, load_graph_types, Graph

from src.search_and_path_algorithms import bfs, printShortestDistance

In [2]:
all_graphs = load_all()

In [3]:
graph_types = load_graph_types()

In [4]:
# for graph in graph_types['Individual']:
#     print(graph.file_path)

increase-n8-2.txt - start is a red node + directed

gnm-10-20-1.txt - start is a red node + non directed

In [14]:
G = all_graphs['grid-10-2.txt']
edges = G.edges

start_ind = G.index[G.names[G.s]]
target_ind = G.index[G.names[G.t]]

if start_ind in G.red or target_ind in G.red:
    print(-1)
    
# for edge in edges:
#     print(G.names[edge[0]], G.names[edge[1]], edge[2])

In [15]:
printShortestDistance(G)

54 32 70 62 30 22 82 94 42 44 

In [7]:
def remove_red(g: Graph, keep_st: bool = True) -> Graph:
    """Return a copy of g without the vertices in R and their edges.

    If keep_st is True, s and t are kept even when they are red.
    Written by Claude (Anthropic)
    """
    keep = [v for v in range(g.n)
            if v not in g.red or (keep_st and v in (g.s, g.t))]
    new_id = {old: new for new, old in enumerate(keep)}   # old index -> new index

    names = [g.names[v] for v in keep]
    adj = [[] for _ in keep]
    edges = []
    for u, v, directed in g.edges:
        if u in new_id and v in new_id:                   # both ends survive
            nu, nv = new_id[u], new_id[v]
            edges.append((nu, nv, directed))
            adj[nu].append(nv)
            if not directed:
                adj[nv].append(nu)

    return Graph(
        names=names,
        index={name: i for i, name in enumerate(names)},
        red={new_id[v] for v in g.red if v in new_id},    # only red s or t can remain
        s=new_id.get(g.s, -1),                            # -1 if s was removed
        t=new_id.get(g.t, -1),
        adj=adj,
        edges=edges,
        directed=g.directed,
        file_path=g.file_path,
    )

In [8]:
G.red, G.s, G.t, G.names

({0, 1, 3, 5, 8}, 0, 1, ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'])

In [16]:
G_without_red = remove_red(G)
G_without_red.edges, G_without_red.names


([(0, 7, False),
  (1, 2, False),
  (1, 4, False),
  (3, 4, False),
  (3, 6, False),
  (5, 6, False),
  (5, 36, False),
  (8, 9, False),
  (8, 47, False),
  (8, 14, False),
  (9, 47, False),
  (10, 11, False),
  (11, 15, False),
  (12, 37, False),
  (13, 14, False),
  (13, 16, False),
  (15, 16, False),
  (17, 28, False),
  (17, 29, False),
  (17, 18, False),
  (17, 20, False),
  (18, 28, False),
  (18, 25, False),
  (19, 30, False),
  (19, 31, False),
  (19, 20, False),
  (19, 22, False),
  (20, 29, False),
  (20, 30, False),
  (21, 32, False),
  (21, 22, False),
  (21, 24, False),
  (22, 31, False),
  (23, 34, False),
  (23, 58, False),
  (23, 33, False),
  (23, 24, False),
  (24, 32, False),
  (24, 33, False),
  (24, 58, False),
  (25, 27, False),
  (25, 26, False),
  (26, 27, False),
  (28, 29, False),
  (29, 30, False),
  (30, 31, False),
  (32, 33, False),
  (33, 34, False),
  (35, 36, False),
  (37, 39, False),
  (38, 46, False),
  (39, 40, False),
  (40, 41, False),
  (41, 42, 

In [17]:
printShortestDistance(G_without_red)

-1


In [11]:
import io, time
from contextlib import redirect_stdout

# Time the pipeline (remove_red + shortest path) on every graph; output is captured so it isn't timed as printing.
timings = []
for name, g in all_graphs.items():
    buf = io.StringIO()
    start = time.perf_counter()
    with redirect_stdout(buf):
        printShortestDistance(remove_red(g))
    timings.append((time.perf_counter() - start, name, g.n, g.m, buf.getvalue().strip()))

for secs, name, n, m, out in sorted(timings, reverse=True):
    print(f"{secs*1000:10.2f} ms  n={n:<8} m={m:<9} {name}\n    {out}")
print(f"\nTotal: {sum(t[0] for t in timings):.2f} s over {len(timings)} graphs")

    138.92 ms  n=80000    m=89999     wall-n-10000.txt
    7 0
     39.61 ms  n=5757     m=163671    rusty-2-5757.txt
    1971 1357 830 4468 4948
     38.85 ms  n=70001    m=80000     wall-z-10000.txt
    7 0
     32.91 ms  n=60002    m=70001     wall-p-10000.txt
    7 0
     29.58 ms  n=5000     m=123432    rusty-2-5000.txt
    1815 742 1795 3801 1536
     24.30 ms  n=4500     m=101086    rusty-2-4500.txt
    1637 676 1620 3433 1390
     18.17 ms  n=4000     m=79432     rusty-2-4000.txt
    1455 839 1143 12 1230
     17.55 ms  n=5757     m=163671    common-2-5757.txt
    -1
     15.17 ms  n=3500     m=61442     rusty-2-3500.txt
    1279 720 994 12 1071
     12.29 ms  n=5000     m=123072    common-2-5000.txt
    -1
     10.83 ms  n=3000     m=45332     rusty-2-3000.txt
    1088 612 844 8 909
      9.46 ms  n=4500     m=100374    common-2-4500.txt
    -1
      7.76 ms  n=4000     m=79124     common-2-4000.txt
    -1
      7.69 ms  n=2500     m=32004     rusty-2-2500.txt
    903 508 695 